In [2]:
#import openmc.data
import pandas as pd
import sqlite3
import numpy as np
import os
from helper_functions import get_A, get_element, get_z, extract_XS_openMC_ace
import matplotlib.pyplot as plt

In [3]:
# DATA PATH CONFIGURATION

# The datasets used in this work are all too large to be included with this repo. 

# openMC's neutron hdf5 data is used by default. 
# This can be replaced with your own ACE files if desired, but you will need to change other functions.
endf7_1_directory = "/global/scratch/users/co_nuclear/endf71x/<symbol>/<ZAID>.710nc"

endf8_directory = "/global/home/groups/co_nuclear/serpent/xsdata/endf8/Lib80x/<symbol>/<ZAID>.800nc"

# x4sqlite1.db location.
# Find it here: https://www-nds.iaea.org/cdroms/
x4sqlite_directory = "sources/x4sqlite1.db"


output_directory = "output/"

## Load from X4Pro and reformat data

In [4]:
# X4 data types 
data_types = {'Reaction': str, 'Projectile': str, 'MT': np.int16, 'Target': str,
              'En': np.float64, 'dEn': np.float64, 'Sig': np.float64, 'dSig': np.float64,
              'fullCode': str, 'YearRef1': np.int16, 'Author1Ini': str, 'Author1': str,
              'DatasetID': str, 'Subent': str, 'Entry': str,}

In [8]:
# Connect to database
con = sqlite3.connect(x4sqlite_directory)

In [10]:
# Extract table data into pandas dataframe
query = """SELECT Reaction, Projectile, En, dEn, Sig, dSig, 
            MT, DatasetID, Entry, Subent, YearRef1, 
            Author1Ini, Author1, Target, fullCode from sig1"""
exfor_df = pd.read_sql_query(query, con, dtype=data_types)
print(exfor_df.head())

  DatasetID  iPoint            fullCode Pointer  Entry    Subent  YearRef1  \
0  10365005       0  0-NN-1(N,TOT),,SIG          10365  10365005      1973   
1  10365005       1  0-NN-1(N,TOT),,SIG          10365  10365005      1973   
2  10365005       2  0-NN-1(N,TOT),,SIG          10365  10365005      1973   
3  10365005       3  0-NN-1(N,TOT),,SIG          10365  10365005      1973   
4  10365005       4  0-NN-1(N,TOT),,SIG          10365  10365005      1973   

   nAuthors Author1Ini Author1  ... sTarg zaTarget1 zaIncident1 outParticles  \
0         6       T.J.  Devlin  ...               1           1                
1         6       T.J.  Devlin  ...               1           1                
2         6       T.J.  Devlin  ...               1           1                
3         6       T.J.  Devlin  ...               1           1                
4         6       T.J.  Devlin  ...               1           1                

  MF  MT           En         dEn     Sig    dSig 

In [11]:
# Close database
con.close()

In [34]:
# Reformat and rename EXFOR data
exfor_df['A'] = exfor_df[['Target']].applymap(get_A)
exfor_df['Element'] = exfor_df[['Target']].applymap(get_element)
exfor_df['Z'] = exfor_df[['Element']].applymap(get_z)
exfor_df['Author'] = exfor_df['Author1Ini'] + exfor_df['Author1']
exfor_df.rename(columns={"En": "Energy", "dEn": "dEnergy", "Subent": "EXFOR_Subentry", "Entry": "EXFOR_Entry", "DatasetID": "Dataset_Number",
                         "YearRef1": "Year", "Sig": "Data", "dSig": "dData"}, inplace=True)
display(exfor_df.head())



/tmp/ipykernel_69382/304028731.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  exfor_df['A'] = exfor_df[['Target']].applymap(get_A)
/tmp/ipykernel_69382/304028731.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  exfor_df['Element'] = exfor_df[['Target']].applymap(get_element)
/tmp/ipykernel_69382/304028731.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  exfor_df['Z'] = exfor_df[['Element']].applymap(get_z)


,Reaction,Projectile,Energy,dEnergy,Data,dData,MT,Dataset_Number,EXFOR_Entry,EXFOR_Subentry,Year,Author1Ini,Author1,Target,fullCode,A,Element,Z,Author
0,"N,TOT",n,247363000.0,15270500.0,0.0202,0.0011,1,10365005,10365,10365005,1973,T.J.,Devlin,NN-1,"0-NN-1(N,TOT),,SIG",1,NN,0,T.J.Devlin
1,"N,TOT",n,278540000.0,15905900.0,0.0221,0.0007,1,10365005,10365,10365005,1973,T.J.,Devlin,NN-1,"0-NN-1(N,TOT),,SIG",1,NN,0,T.J.Devlin
2,"N,TOT",n,310939000.0,16493300.0,0.0233,0.0008,1,10365005,10365,10365005,1973,T.J.,Devlin,NN-1,"0-NN-1(N,TOT),,SIG",1,NN,0,T.J.Devlin
3,"N,TOT",n,344468000.0,17036100.0,0.0249,0.0005,1,10365005,10365,10365005,1973,T.J.,Devlin,NN-1,"0-NN-1(N,TOT),,SIG",1,NN,0,T.J.Devlin
4,"N,TOT",n,379042000.0,17537500.0,0.0266,0.0003,1,10365005,10365,10365005,1973,T.J.,Devlin,NN-1,"0-NN-1(N,TOT),,SIG",1,NN,0,T.J.Devlin


In [35]:
relative_unc = exfor_df["dData"]/exfor_df["Data"]
relative_unc = relative_unc.replace(np.inf, np.nan)
relative_unc = relative_unc.dropna()
Q90_ERROR = np.quantile(relative_unc, 0.90)
print(f"90th Quantile Relative Uncertainty: {round(Q90_ERROR*100, 3)}%")

90th Quantile Relative Uncertainty: 36.145%


In [61]:
# Get list of unique isotopes and write it for NuGrade
all_isotopes = exfor_df.drop_duplicates(subset=['Z', 'A'])[["Z", "A", "Element"]]
reaction_summary_list = []
neutron_reaction_channels = {"N,TOT":1, "N,EL":2, "N,INL":4, "N,G":102}
proton_reaction_channels = {"P,EL":2, "P,INL":103, "P,G":102}

In [64]:
def compute_channel_data(eval_path, evaluation, channel_data, mt, energy_values, output_directory, quantile=0.9, fallback_error=Q90_ERROR):
    if not os.path.exists(output_directory + "evals/" + evaluation):
        os.mkdir(output_directory + "evals/" + evaluation)
    found_eval = False
    # Check if evaluation exists, and then check if evaluation has this MT
    if os.path.isfile(eval_path):
        try:
            interp_xs, grid_energy, grid_xs = extract_XS_openMC_ace(eval_path, mt, energy_values)
            found_eval = True
        except KeyError:
            found_eval = False
        except AssertionError:
            found_eval = False
            print(f"AssertionError for {eval_path}")
    else:
        found_eval = False

    # fill missing values for uncertainty
    channel_data_imputed = channel_data.copy()

    # Use 90% percentile within this nuclide/reaction's data (max 1000% error to handle likely input errors)
    if len(channel_data.dropna()) > 0:
        relative_unc = channel_data["dData"]/channel_data["Data"]
        relative_unc = relative_unc.replace(np.inf, np.nan)
        relative_unc = relative_unc.dropna()
        assumed_error_fraction = np.min([np.quantile(relative_unc, 0.90), 10])
    # If no data in this nuclide/reaction's data has any uncertainty, fall back a global assumption
    else:
        assumed_error_fraction = fallback_error
    channel_data_imputed["dData"] = channel_data_imputed["dData"].fillna(channel_data_imputed["Data"]*assumed_error_fraction)
    channel_data["dData_assumed"] = channel_data_imputed["dData"]

    # If evaluation exists, compute relative uncertainty and chi squared relative to it
    if found_eval:
        channel_data[evaluation] = interp_xs

        channel_data[evaluation + '_chi_squared'] = ((channel_data_imputed['Data'] - channel_data[evaluation]) ** 2) / \
                                                    channel_data_imputed['dData']
        channel_data[evaluation + '_relative_error'] = (channel_data['Data'] - channel_data[evaluation]) / channel_data[
            evaluation] * 100
        channel_data[evaluation + '_relative_error'] = channel_data[evaluation + '_relative_error'].replace(np.inf, np.nan)
        
        file_name = output_directory + "evals/" + evaluation+"/"+ f"{proj}_{Z}_{A}_{reaction}_grid_data.csv"
        
        grid_df = pd.DataFrame({"grid_energy(eV)":grid_energy, "grid_xs(b)":grid_xs})
        grid_df.to_csv(file_name, na_rep='nan')

    # If not, set to nan indicating there isn't one
    else:
        f"Reading {evaluation} failed for {symbol}{reaction}."
        channel_data[evaluation] = np.nan
        channel_data[evaluation + "_relative_error"] = np.nan
        channel_data[evaluation + "_chi_squared"] = np.nan
        print(f"{eval_path} not found. Evaluation does not exist?")
    return channel_data, found_eval

## Construct measurement data table one reaction channel at a time

In [69]:
conn = sqlite3.connect('output/nugrade_data.db')

have_written = False

# Iterate through unique isotopes
for index, row in all_isotopes.iterrows():
    Z = row.iloc[0]
    A = row.iloc[1]
    symbol = row.iloc[2]
    symbol_caps = symbol[0].upper() + symbol[1:]
    if Z < 1 or A < 1:
        continue
    ZAID = str(Z) + str(A).zfill(3)



    
    for reaction in neutron_reaction_channels.keys():
        print(reaction)
        mt = neutron_reaction_channels[reaction]
        proj = reaction[0].lower()
        channel_data = exfor_df.loc[(exfor_df['Z'] == Z)
                                    & (exfor_df['A'] == A)
                                    & (exfor_df['Reaction'] == reaction)]
        channel_data = channel_data
        if len(channel_data.index) == 0:
            continue
        else:
            channel_summary = [Z,A,symbol,proj,mt, reaction]
        channel_data = channel_data.sort_values(by=['Energy'])
        energy_values = channel_data['Energy'].to_numpy()

        
        eval_path = endf7_1_directory.replace("<symbol>", symbol_caps).replace("<ZAID>",ZAID)
        evaluation = "endf7-1"
        channel_data, eval_exists = compute_channel_data(eval_path, evaluation, channel_data, mt, energy_values, output_directory)
        channel_summary += [eval_exists]
        
        eval_path = endf8_directory.replace("<symbol>", symbol_caps).replace("<ZAID>",ZAID)
        evaluation = "endf8"
        channel_data, eval_exists = compute_channel_data(eval_path, evaluation, channel_data, mt, energy_values, output_directory)
        channel_summary += [eval_exists]
        
        file_name = output_directory + f"{proj}_{Z}_{A}_{reaction}.csv"
        if have_written:
            channel_data.to_sql('measurements', conn, if_exists='append', index=False)
        else:
            channel_data.to_sql('measurements', conn, if_exists='replace', index=False)
            have_written = True
        #channel_data.to_csv(file_name, na_rep='nan')
        reaction_summary_list += [channel_summary]

conn.close()

N,TOT


/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/energy_distribution.py:1244: UserWarning: Interpolation scheme for continuous tabular distribution is not histogram or linear-linear.
  warn("Interpolation scheme for continuous tabular distribution "
/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/energy_distribution.py:1244: UserWarning: Interpolation scheme for continuous tabular distribution is not histogram or linear-linear.
  warn("Interpolation scheme for continuous tabular distribution "


N,EL


/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/energy_distribution.py:1244: UserWarning: Interpolation scheme for continuous tabular distribution is not histogram or linear-linear.
  warn("Interpolation scheme for continuous tabular distribution "
/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/energy_distribution.py:1244: UserWarning: Interpolation scheme for continuous tabular distribution is not histogram or linear-linear.
  warn("Interpolation scheme for continuous tabular distribution "


N,INL


...

N,EL


/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/neutron.py:601: UserWarning: Photon production is present for MT=28 but no cross section is given.
  warn('Photon production is present for MT={} but no '
/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/neutron.py:601: UserWarning: Photon production is present for MT=32 but no cross section is given.
  warn('Photon production is present for MT={} but no '
/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/neutron.py:601: UserWarning: Photon production is present for MT=28 but no cross section is given.
  warn('Photon production is present for MT={} but no '
/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/neutron.py:601: UserWarning: Photon production is present for MT=32 but no cross section is given.
  warn('Photon production is present for MT={} but no '


N,INL
N,G


/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/neutron.py:601: UserWarning: Photon production is present for MT=28 but no cross section is given.
  warn('Photon production is present for MT={} but no '
/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/neutron.py:601: UserWarning: Photon production is present for MT=32 but no cross section is given.
  warn('Photon production is present for MT={} but no '
/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/neutron.py:601: UserWarning: Photon production is present for MT=28 but no cross section is given.
  warn('Photon production is present for MT={} but no '
/global/scratch/users/ikolaja/conda/python3_11_6/lib/python3.11/site-packages/openmc/data/neutron.py:601: UserWarning: Photon production is present for MT=32 but no cross section is given.
  warn('Photon production is present for MT={} but no '


N,TOT
N,EL
N,INL
N,G
N,TOT
/global/scratch/users/co_nuclear/endf71x/Yb/70168.710nc not found. Evaluation does not exist?
N,EL
N,INL
N,G
/global/scratch/users/co_nuclear/endf71x/Yb/70168.710nc not found. Evaluation does not exist?
N,TOT
/global/scratch/users/co_nuclear/endf71x/Yb/70170.710nc not found. Evaluation does not exist?
N,EL
N,INL
N,G
/global/scratch/users/co_nuclear/endf71x/Yb/70170.710nc not found. Evaluation does not exist?
N,TOT
/global/scratch/users/co_nuclear/endf71x/Yb/70171.710nc not found. Evaluation does not exist?
N,EL
N,INL
N,G
/global/scratch/users/co_nuclear/endf71x/Yb/70171.710nc not found. Evaluation does not exist?
N,TOT
/global/scratch/users/co_nuclear/endf71x/Yb/70172.710nc not found. Evaluation does not exist?
N,EL
N,INL
N,G
/global/scratch/users/co_nuclear/endf71x/Yb/70172.710nc not found. Evaluation does not exist?
N,TOT
/global/scratch/users/co_nuclear/endf71x/Yb/70173.710nc not found. Evaluation does not exist?
N,EL
N,INL
N,G
/global/scratch/users/co_nuc

In [70]:
summary_list_headers = ["Z", "A", "Symbol", "Projectile", "Reaction", "MT", "has_endf7-1", "has_endf8"]
summary_df = pd.DataFrame(reaction_summary_list, columns = summary_list_headers)
display(summary_df)
file_name = output_directory + f"all_reactions.csv"
summary_df.to_csv(file_name)

,Z,A,Symbol,Projectile,Reaction,MT,has_endf7-1,has_endf8
0,1,1,H,n,1,"N,TOT",True,True
1,1,1,H,n,2,"N,EL",True,True
2,1,1,H,n,4,"N,INL",False,False
3,1,1,H,n,102,"N,G",True,True
4,1,2,H,n,1,"N,TOT",True,True
...,...,...,...,...,...,...,...,...
825,98,250,Cf,n,1,"N,TOT",True,True
826,98,250,Cf,n,102,"N,G",True,True
827,98,251,Cf,n,1,"N,TOT",True,True
828,98,252,Cf,n,102,"N,G",True,True


In [71]:
measurement_df = pd.read_sql('SELECT * FROM measurements', conn)


## Compute Subentry level and Entry level metrics

In [5]:
conn = sqlite3.connect('output/nugrade_data.db')
measurement_df = pd.read_sql('SELECT * FROM measurements', conn)

In [6]:
subentries = []
i = 0
for exfor_subentry in measurement_df["EXFOR_Subentry"].unique():
    subentry_measurement_df = measurement_df[measurement_df["EXFOR_Subentry"] == exfor_subentry]
    if i % 100 == 0:
        print(f"Subentry {i}")
    subentry_dict = {
        "Z": subentry_measurement_df["Z"].iloc[0],
        "A": subentry_measurement_df["A"].iloc[0],
        "MT": subentry_measurement_df["MT"].iloc[0],
        "Reaction": subentry_measurement_df["Reaction"].iloc[0],
        "Element": subentry_measurement_df["Element"].iloc[0],
        "EXFOR_Subentry": subentry_measurement_df["EXFOR_Subentry"].iloc[0],
        "EXFOR_Entry": subentry_measurement_df["EXFOR_Entry"].iloc[0],
        "Dataset_Number": subentry_measurement_df["Dataset_Number"].iloc[0],
        "E_min": subentry_measurement_df["Energy"].min(),
        "E_max": subentry_measurement_df["Energy"].max(),
        "E_median": subentry_measurement_df["Energy"].median(),
        "num_measurements": len(subentry_measurement_df),
        "num_measurements_uncertainty": len(subentry_measurement_df)-sum(subentry_measurement_df["dData"].isna())
    }
    subentries.append(subentry_dict)
    i += 1
subentry_df = pd.DataFrame(subentries)
display(subentry_df[:-10])
subentry_df.to_sql('subentries', conn, if_exists='replace', index=False)

Subentry 0
Subentry 100
Subentry 200
Subentry 300
Subentry 400
Subentry 500
Subentry 600
Subentry 700
Subentry 800
Subentry 900
Subentry 1000
Subentry 1100
Subentry 1200
Subentry 1300
Subentry 1400
Subentry 1500
Subentry 1600
Subentry 1700
Subentry 1800
Subentry 1900
Subentry 2000
Subentry 2100
Subentry 2200
Subentry 2300
Subentry 2400
Subentry 2500
Subentry 2600
Subentry 2700
Subentry 2800
Subentry 2900
Subentry 3000
Subentry 3100
Subentry 3200
Subentry 3300
Subentry 3400
Subentry 3500
Subentry 3600
Subentry 3700
Subentry 3800
Subentry 3900
Subentry 4000
Subentry 4100
Subentry 4200
Subentry 4300
Subentry 4400
Subentry 4500
Subentry 4600
Subentry 4700
Subentry 4800
Subentry 4900
Subentry 5000
Subentry 5100
Subentry 5200
Subentry 5300
Subentry 5400
Subentry 5500
Subentry 5600
Subentry 5700
Subentry 5800
Subentry 5900
Subentry 6000
Subentry 6100


,Z,A,MT,Reaction,Element,EXFOR_Subentry,EXFOR_Entry,Dataset_Number,E_min,E_max,E_median,num_measurements,num_measurements_uncertainty
0,1,1,1,"N,TOT",H,11432005,11432,11432005,0.002819,96.7500,0.250400,49,0
1,1,1,1,"N,TOT",H,11150002,11150,11150002,0.003300,1830.0000,0.213500,62,0
2,1,1,1,"N,TOT",H,21146010,21146,21146010,0.022570,0.5400,0.152500,64,64
3,1,1,1,"N,TOT",H,21146007,21146,21146007,0.022570,0.3311,0.055935,30,30
4,1,1,1,"N,TOT",H,21146003,21146,21146003,0.022570,0.4086,0.058180,31,31
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6178,98,249,1,"N,TOT",Cf,10724009,10724,10724009,0.019777,18858.0000,16.897000,1660,1660
6179,98,249,1,"N,TOT",Cf,40735002,40735,40735002,0.025000,0.0250,0.025000,1,1
6180,98,249,1,"N,TOT",Cf,10724010,10724,10724010,0.383550,31732.0000,31.616000,3395,3395
6181,98,249,1,"N,TOT",Cf,10724011,10724,10724011,0.383550,31663.0000,30.794500,3490,3490


In [7]:
measurement_df.columns

Index(['Reaction', 'Projectile', 'Energy', 'dEnergy', 'Data', 'dData', 'MT',
       'Dataset_Number', 'EXFOR_Entry', 'EXFOR_Subentry', 'Year', 'Author1Ini',
       'Author1', 'Target', 'fullCode', 'A', 'Element', 'Z', 'Author',
       'dData_assumed', 'endf7-1', 'endf7-1_chi_squared',
       'endf7-1_relative_error', 'endf8', 'endf8_chi_squared',
       'endf8_relative_error'],
      dtype='object')

In [32]:
entries = []
i = 0
for exfor_entry in measurement_df["EXFOR_Entry"].unique():
    entry_measurement_df = measurement_df[measurement_df["EXFOR_Entry"] == exfor_entry]
    if i % 100 == 0:
        print(f"entry {i}")
    entry_dict = {
        "Z_Values": ",".join(entry_measurement_df["Z"].unique().astype(str)),
        "Elements": ",".join(entry_measurement_df["Element"].unique()),
        "A_Values": ",".join(entry_measurement_df["A"].unique().astype(str)),
        "MT_Codes": ",".join(entry_measurement_df["MT"].unique().astype(str)),
        "Reactions": "-".join(entry_measurement_df["Reaction"].unique().astype(str)),
        "EXFOR_Entry": entry_measurement_df["EXFOR_Entry"].iloc[0],
        "Num_Subentries": len(entry_measurement_df["EXFOR_Subentry"].unique()),
        "Num_Measurements": len(entry_measurement_df),
        "Num_Missing_Uncertainty":  sum(entry_measurement_df["dData"].isna()),
        "Uncertainty_Complete":  not entry_measurement_df["dData"].isna().any()
    }
    entries.append(entry_dict)
    i += 1
entry_df = pd.DataFrame(entries)
display(entry_df[:-10])

entry 0
entry 100
entry 200
entry 300
entry 400
entry 500
entry 600
entry 700
entry 800
entry 900
entry 1000
entry 1100
entry 1200
entry 1300
entry 1400
entry 1500
entry 1600
entry 1700
entry 1800
entry 1900
entry 2000
entry 2100


,Z_Values,Elements,A_Values,MT_Codes,Reactions,EXFOR_Entry,Num_Subentries,Num_Measurements,Num_Missing_Uncertainty,Uncertainty_Complete
0,1,H,1,1,"N,TOT",11432,2,51,49,False
1,1,H,1,1,"N,TOT",11150,1,62,62,False
2,1,H,1,1,"N,TOT",21146,15,417,0,True
3,"1,13,83","H,Al,Bi","1,2,27,209",1,"N,TOT",12589,4,6,3,False
4,1,H,1,1,"N,TOT",10139,1,64,0,True
...,...,...,...,...,...,...,...,...,...,...
2178,96,Cm,"244,245",1,"N,TOT",10271,2,2,0,True
2179,96,Cm,244,1,"N,TOT",14247,1,1860,0,True
2180,96,Cm,244,1,"N,TOT",12570,1,328,0,True
2181,"96,98","Cm,Cf","244,245,246,247,248,250",102,"N,G",40486,6,6,2,False


In [34]:
entry_df.to_sql('entries', conn, if_exists='replace', index=False)
conn.close()

In [36]:
# Check completeness at subentry level
subentry_completeness = measurement_df.groupby("EXFOR_Subentry").apply(
    lambda x: x["dData"].isna().mean()
)
print(subentry_completeness.value_counts(bins=10))

(-0.002, 0.1]    5050
(0.9, 1.0]       1029
(0.8, 0.9]         45
(0.7, 0.8]         21
(0.4, 0.5]         12
(0.5, 0.6]          8
(0.6, 0.7]          8
(0.1, 0.2]          7
(0.3, 0.4]          7
(0.2, 0.3]          6
Name: count, dtype: int64
